# TTA evaluation — histopath baseline
Inference-only: re-scores the trained baseline checkpoint with test-time augmentation (avg over 8 flip/rotation views). No retraining.

## Stage code + data (same as the main runner)

In [ ]:
import os, sys, glob, shutil, gzip, zipfile, subprocess
ON_KAGGLE = os.path.exists('/kaggle/input')
print('Kaggle' if ON_KAGGLE else 'Colab/local')

# Diagnostic: show what's actually mounted (self-documents any future failure).
if ON_KAGGLE:
    for d in sorted(glob.glob('/kaggle/input/*')):
        print(' input:', d, '->', sorted(os.path.basename(x) for x in glob.glob(d + '/*'))[:8])

# `kaggle kernels push` uploads only THIS notebook. Bring in src/ + configs from the
# attached dataset (no internet needed). Prefer unzipping code.zip (clean WSI .gz);
# fall back to an already-extracted tree, then git clone (only works with internet).
REPO_URL = 'https://github.com/aRealGem/histopath-cancer-detection'
WORK = '/kaggle/working/repo'
if not os.path.exists('src/train.py'):
    os.makedirs(WORK, exist_ok=True)
    czip = glob.glob('/kaggle/input/**/code.zip', recursive=True)
    extracted = glob.glob('/kaggle/input/**/src/train.py', recursive=True)
    if czip:
        with zipfile.ZipFile(czip[0]) as z:
            z.extractall(WORK)
        print('staged from code.zip:', czip[0])
    elif extracted:
        root = os.path.dirname(os.path.dirname(extracted[0]))
        shutil.copytree(root, WORK, dirs_exist_ok=True)
        print('staged from extracted dataset:', root)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK], check=True)
        print('staged from git clone')
    os.chdir(WORK)

# WSI map: if Kaggle decompressed the shipped .gz into a folder, re-materialize it
# at the config path. (When we unzip code.zip ourselves, the .gz is already intact.)
if not os.path.exists('data/wsi/patch_id_wsi_full.csv.gz'):
    hits = [h for h in glob.glob('/kaggle/input/**/*wsi*full*.csv', recursive=True) if os.path.isfile(h)]
    if hits:
        os.makedirs('data/wsi', exist_ok=True)
        with open(hits[0], 'rb') as fi, gzip.open('data/wsi/patch_id_wsi_full.csv.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        print('WSI map normalized from', hits[0])

# Offline ImageNet weights: internet is off, so cache the bundled backbone weights.
kmodels = os.path.expanduser('~/.keras/models')
os.makedirs(kmodels, exist_ok=True)
wts = glob.glob('/kaggle/input/**/weights_mobilenet_v3_small_224_1.0_float_no_top_v2.h5', recursive=True)
if wts:
    shutil.copy(wts[0], os.path.join(kmodels, os.path.basename(wts[0])))
    print('cached backbone weights offline')
else:
    print('WARNING: bundled MobileNetV3 weights not found; backbone would need internet.')

# Auto-detect the competition mount and write the real data.root into the configs.
import yaml
cands = glob.glob('/kaggle/input/**/train_labels.csv', recursive=True)
if cands:
    DATA_ROOT = os.path.dirname(cands[0])
    print('data.root ->', DATA_ROOT)
    for cf in glob.glob('configs/*.yaml'):
        conf = yaml.safe_load(open(cf))
        if not isinstance(conf, dict) or 'data' not in conf:
            continue  # skip meta-configs like sweep.yaml (no data block)
        conf['data']['root'] = DATA_ROOT
        yaml.safe_dump(conf, open(cf, 'w'), sort_keys=False)
else:
    print('WARNING: train_labels.csv not found under /kaggle/input.')

sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd(), '| src:', os.path.exists('src/train.py'),
      '| wsi:', os.path.exists('data/wsi/patch_id_wsi_full.csv.gz'))

In [ ]:
# Kaggle images already ship cv2, pyyaml, scikit-learn — installing again is slow
# and can conflict. Only install on Colab/local.
if not ON_KAGGLE:
    get_ipython().system('pip -q install opencv-python-headless pyyaml scikit-learn')
else:
    print('Kaggle: cv2/pyyaml/sklearn preinstalled; skipping pip install')

## Run TTA eval (val no-TTA vs TTA) + write TTA submission

In [ ]:
# Locate the baseline checkpoint (mounted from the full-train kernel via kernel_sources)
import glob
for d in sorted(glob.glob('/kaggle/input/*')): print(' input:', d)
cks = glob.glob('/kaggle/input/**/best.keras', recursive=True)
assert cks, 'best.keras not found in kernel inputs — check kernel_sources'
MODEL = cks[0]
print('using model:', MODEL)
!python -m src.tta_eval --config configs/baseline.yaml --model {MODEL} --test